# 🐾 Animal Detection — Experiments & Analysis

This notebook is the **companion to the main GUI app**.
Here we:
- Explore the COCO dataset animal classes
- Test YOLOv8 on sample images and visualize results
- Compare nano vs small model accuracy
- Plot confidence distributions
- Show confusion-style breakdowns by species

---
> **Internship Task:** Animal Detection Model — classification + carnivore highlighting


In [ ]:
# Install dependencies (run once)
# !pip install ultralytics opencv-python matplotlib seaborn pillow

import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')

# Set consistent plot style
plt.style.use('dark_background')
sns.set_palette('husl')

print('Libraries loaded ✓')

## 1. Load Model & Define Animal Taxonomy

In [ ]:
# Load YOLOv8 nano — lightest model, good for quick experiments
model_nano = YOLO('yolov8n.pt')
model_small = YOLO('yolov8s.pt')  # we'll compare these later

# COCO animal class IDs (relevant subset)
ANIMAL_IDS = {14: 'bird', 15: 'cat', 16: 'dog', 17: 'horse',
              18: 'sheep', 19: 'cow', 20: 'elephant', 21: 'bear',
              22: 'zebra', 23: 'giraffe'}

CARNIVORES = {'cat', 'bear', 'lion', 'tiger', 'eagle', 'hawk', 'bird', 'dog'}

print(f'COCO animal classes: {list(ANIMAL_IDS.values())}')
print(f'Carnivores flagged: {sorted(CARNIVORES)}')

## 2. Single Image Detection Test

In [ ]:
# Test on a sample image
# Replace with your own image path
TEST_IMAGE = 'samples/animals_test.jpg'  # put any animal image here

results = model_nano(TEST_IMAGE, conf=0.45)

# Plot the result
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Original
orig = Image.open(TEST_IMAGE)
axes[0].imshow(orig)
axes[0].set_title('Original Image', fontsize=12)
axes[0].axis('off')

# Annotated
annotated = results[0].plot()
axes[1].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
axes[1].set_title('YOLOv8 Detections', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('outputs/single_detection.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/single_detection.png')

## 3. Model Comparison — Nano vs Small

In [ ]:
import time

def benchmark_model(model, image_path, runs=5):
    """Benchmark a model — returns avg inference time and detection count."""
    times = []
    for _ in range(runs):
        t0 = time.time()
        results = model(image_path, conf=0.45, verbose=False)
        times.append(time.time() - t0)
    
    n_detections = len(results[0].boxes) if results[0].boxes else 0
    return {
        'avg_time_ms': round(np.mean(times) * 1000, 1),
        'std_ms': round(np.std(times) * 1000, 1),
        'detections': n_detections,
    }

# Run benchmarks
bench_nano  = benchmark_model(model_nano, TEST_IMAGE)
bench_small = benchmark_model(model_small, TEST_IMAGE)

print('Model Comparison:')
print(f'  YOLOv8n (nano):  {bench_nano["avg_time_ms"]} ms avg | {bench_nano["detections"]} detections')
print(f'  YOLOv8s (small): {bench_small["avg_time_ms"]} ms avg | {bench_small["detections"]} detections')

# Bar chart comparison
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

models = ['YOLOv8n (nano)', 'YOLOv8s (small)']
times = [bench_nano['avg_time_ms'], bench_small['avg_time_ms']]
dets  = [bench_nano['detections'], bench_small['detections']]

axes[0].bar(models, times, color=['#4ecca3', '#e94560'])
axes[0].set_title('Inference Time (ms) — lower is better')
axes[0].set_ylabel('ms')

axes[1].bar(models, dets, color=['#4ecca3', '#e94560'])
axes[1].set_title('Detections Found — higher = more complete')
axes[1].set_ylabel('count')

plt.suptitle('YOLOv8 Nano vs Small — Speed vs Accuracy Tradeoff', fontsize=13)
plt.tight_layout()
plt.savefig('outputs/model_comparison.png', dpi=150)
plt.show()

## 4. Confidence Distribution by Species

In [ ]:
import glob
from collections import defaultdict

# Run on all sample images and collect confidence scores per species
# Add your test images to the samples/ folder
image_paths = glob.glob('samples/*.jpg') + glob.glob('samples/*.png')

species_confidences = defaultdict(list)

for path in image_paths:
    results = model_nano(path, conf=0.3, verbose=False)
    for box in (results[0].boxes or []):
        cls_id = int(box.cls[0])
        if cls_id in ANIMAL_IDS:
            label = ANIMAL_IDS[cls_id]
            species_confidences[label].append(float(box.conf[0]))

if species_confidences:
    fig, ax = plt.subplots(figsize=(10, 5))
    data = [species_confidences[s] for s in species_confidences]
    labels = list(species_confidences.keys())
    colors = ['#ff6b6b' if s in CARNIVORES else '#4ecca3' for s in labels]
    
    bp = ax.boxplot(data, patch_artist=True, labels=labels)
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    
    ax.set_title('Detection Confidence by Species\n(🔴 red = carnivore, 🟢 green = herbivore)')
    ax.set_ylabel('Confidence Score')
    ax.set_ylim(0, 1)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.savefig('outputs/confidence_by_species.png', dpi=150)
    plt.show()
else:
    print('No animal images found in samples/ — add some .jpg files and re-run!')

## 5. Carnivore vs Herbivore Detection Breakdown

In [ ]:
# Pie chart: how many carnivores vs herbivores were detected across all test images
all_labels = [label for label, confs in species_confidences.items() for _ in confs]

carnivore_count  = sum(1 for l in all_labels if l in CARNIVORES)
herbivore_count  = len(all_labels) - carnivore_count

if all_labels:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    # Pie chart
    axes[0].pie(
        [carnivore_count, herbivore_count],
        labels=['Carnivores 🔴', 'Herbivores 🟢'],
        colors=['#e94560', '#4ecca3'],
        autopct='%1.1f%%',
        startangle=140,
        textprops={'color': 'white', 'fontsize': 11}
    )
    axes[0].set_title('Carnivore vs Herbivore Ratio')

    # Bar chart: detections per species
    from collections import Counter
    counts = Counter(all_labels)
    species = list(counts.keys())
    vals = list(counts.values())
    bar_colors = ['#ff6b6b' if s in CARNIVORES else '#4ecca3' for s in species]
    
    axes[1].barh(species, vals, color=bar_colors)
    axes[1].set_title('Detections per Species')
    axes[1].set_xlabel('Number of detections')
    
    plt.suptitle('Animal Detection Summary', fontsize=14)
    plt.tight_layout()
    plt.savefig('outputs/detection_summary.png', dpi=150)
    plt.show()
else:
    print('No detections yet — run on sample images first.')

## 6. Summary & Next Steps

### What we built:
- **YOLOv8-based animal detector** running on COCO pretrained weights
- **Carnivore tagging** — red boxes, popup alert, CSV logging
- **Full GUI app** (see `app.py`) with image + video + webcam support
- **Model comparison** — nano vs small speed/accuracy tradeoff

### Possible extensions:
- Fine-tune YOLOv8 on a custom wildlife dataset (e.g. iNaturalist)
- Add species count tracking over time (for wildlife monitoring)
- Integrate GPS metadata from image EXIF for location-aware alerts
- Train a separate classifier specifically for carnivore/herbivore distinction
  (YOLOv8 COCO doesn't include lions, tigers by default)

### Limitations:
- COCO only has 10 animal classes — many wildlife species won't be detected
- Carnivore list is hardcoded — a real system should use a taxonomy API
- No tracking between frames (could add DeepSORT for video)
